# Database provisioning — docker-host smoke analysis

Post-hoc analysis of the guarded `docker-host` smoke after the scorer repairs
(Claude tool-call lifecycle merge and loopback libSQL URI/operation support).

- Experiment: `database-provisioning-v2`
- Run request: `01KZVHRWZ2YYHK1TKSEFFE4EM4`
- Matrix: 3 prompts × Codex/Terra, Claude/Sonnet, Cursor/Grok 4.5 × `docker-host` × n=2 (18 trials)
- Date: 2026-08-12

This notebook queries AX with `ax experiment query <id> sql` (ClickHouse SQL →
NDJSON → pandas). That is the documented query surface; the Jupyter kernel does
not inherit `AX_API_KEY`. If `ax` is not on `PATH`, it falls back to HTTP using
`AX_API_KEY` / `AXP_API_KEY`, a repo `.env`, or `~/.fiveonefour/axp/credentials.toml`.

Queries return derived metrics and boolean evidence, not raw URIs or tool
payloads, so generated credentials stay out of the notebook output.

Install analysis dependencies into the active kernel (the project `.venv` if that is selected). Then re-run the next cell.

`ax auth status` should show a signed-in org. Offline alternative: `ax run download 01KZVHRWZ2YYHK1TKSEFFE4EM4`, then set `AX_PARQUET_DIR` to the resulting `.axp/downloads/<id>/` directory.

In [ ]:
%pip install requests pandas matplotlib numpy python-dotenv

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

EXPERIMENT_ID = "database-provisioning-v2"
RUN_REQUEST_ID = "01KZVHRWZ2YYHK1TKSEFFE4EM4"
AX_API_BASE = os.environ.get("AX_API_BASE", "https://app.514.ax").rstrip("/")
AX_PARQUET_DIR = os.environ.get("AX_PARQUET_DIR", "")
CLI_CREDENTIALS_PATH = Path.home() / ".fiveonefour/axp/credentials.toml"

DATABASE_ORDER = ["mongodb", "postgresql", "sqlite"]
AGENT_ORDER = ["codex", "claude", "cursor"]
DISPLAY_NAMES = {
    "mongodb": "MongoDB",
    "postgresql": "PostgreSQL",
    "sqlite": "SQLite",
    "codex": "Codex + Terra",
    "claude": "Claude + Sonnet",
    "cursor": "Cursor + Grok 4.5",
}
COLORS = {
    "mongodb": "#00A35C",
    "postgresql": "#336791",
    "sqlite": "#8C8C8C",
    "codex": "#111827",
    "claude": "#D97706",
    "cursor": "#7C3AED",
}

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)


def _load_dotenv() -> None:
    try:
        from dotenv import load_dotenv
    except ImportError:
        return
    here = Path.cwd().resolve()
    for directory in [here, *here.parents]:
        env_file = directory / ".env"
        if env_file.is_file():
            load_dotenv(env_file)
            return


def _credentials_from_cli_store() -> dict[str, str]:
    """Read key names from the AX CLI store. Never print values."""
    if not CLI_CREDENTIALS_PATH.is_file():
        return {}
    values: dict[str, str] = {}
    for line in CLI_CREDENTIALS_PATH.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or stripped.startswith("["):
            continue
        if "=" not in stripped:
            continue
        key, value = stripped.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


_load_dotenv()
AX_ORG_ID = os.environ.get("AX_ORG_ID", "org_3FJv0fCt81VIGxO61n6HNbYLnHl")


def _api_key() -> str:
    key = os.environ.get("AX_API_KEY") or os.environ.get("AXP_API_KEY")
    if not key:
        key = _credentials_from_cli_store().get("api_key")
    if not key:
        raise RuntimeError(
            "No AX API key in AX_API_KEY / AXP_API_KEY, a repo .env, or "
            f"{CLI_CREDENTIALS_PATH}. Run `ax auth login` or create a token at "
            "https://app.514.ax/account/api-keys."
        )
    return key


def _parse_query_response(response: requests.Response) -> pd.DataFrame:
    text = response.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        payload = response.json()
    except ValueError:
        rows = [json.loads(line) for line in text.splitlines() if line.strip()]
        return pd.DataFrame(rows)

    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        for key in ("rows", "data", "result", "results"):
            value = payload.get(key)
            if isinstance(value, list):
                return pd.DataFrame(value)
        if payload.get("sql") or payload.get("error"):
            raise RuntimeError(payload.get("error") or payload)
    raise RuntimeError(f"Unexpected AX query response shape: {type(payload).__name__}")


def ax_http_query(
    sql: str,
    *,
    experiment_id: str = EXPERIMENT_ID,
    run_id: str | None = None,
    limit: int = 10_000,
) -> pd.DataFrame:
    """Run ClickHouse SQL through the AX HTTP query API."""
    path = (
        f"/api/v1/runs/{run_id}/query"
        if run_id
        else f"/api/v1/experiments/{experiment_id}/query"
    )
    headers = {
        "Authorization": f"Bearer {_api_key()}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    if AX_ORG_ID:
        headers["X-Org-Id"] = AX_ORG_ID
    response = requests.post(
        f"{AX_API_BASE}{path}",
        headers=headers,
        json={"sql": sql, "limit": limit, "format": "json"},
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(
            f"AX query failed ({response.status_code}): {response.text[:500]}"
        )
    return _parse_query_response(response)


def ax_cli_query(sql: str, *, limit: int = 10_000) -> pd.DataFrame:
    """Run ClickHouse SQL through `ax experiment query <id> sql`."""
    ax = shutil.which("ax")
    if not ax:
        raise FileNotFoundError("ax CLI is not on PATH")
    command = [
        ax,
        "experiment",
        "query",
        EXPERIMENT_ID,
        "sql",
        sql,
        "--format",
        "json",
        "--limit",
        str(limit),
    ]
    if AX_ORG_ID:
        command.extend(["--org", AX_ORG_ID])
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"ax experiment query failed ({result.returncode}): {result.stderr.strip()[:500]}"
        )
    rows: list[dict] = []
    for line in result.stdout.splitlines():
        stripped = line.strip()
        if not stripped or stripped[0] not in "{[":
            continue
        try:
            parsed = json.loads(stripped)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, list):
            rows.extend(parsed)
        elif isinstance(parsed, dict):
            rows.append(parsed)
    return pd.DataFrame(rows)


def ax_parquet_query(sql: str) -> pd.DataFrame:
    """Query downloaded Parquet with DuckDB. Table names are file stems."""
    try:
        import duckdb
    except ImportError as exc:
        raise RuntimeError("Install duckdb to query AX_PARQUET_DIR.") from exc

    root = Path(AX_PARQUET_DIR).expanduser()
    files = sorted(root.rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files under {root}")

    con = duckdb.connect()
    for path in files:
        con.execute(
            f"CREATE OR REPLACE VIEW {path.stem} AS SELECT * FROM read_parquet(?)",
            [str(path)],
        )
    return con.execute(sql).df()


def ax_query(sql: str, **kwargs) -> pd.DataFrame:
    if AX_PARQUET_DIR:
        return ax_parquet_query(sql)
    if shutil.which("ax"):
        return ax_cli_query(sql, limit=kwargs.get("limit", 10_000))
    return ax_http_query(sql, **kwargs)


def _with_database_column(frame: pd.DataFrame) -> pd.DataFrame:
    if "database" not in frame.columns and "prompt_id" in frame.columns:
        frame = frame.rename(columns={"prompt_id": "database"})
    return frame


print(f"Experiment: {EXPERIMENT_ID}")
print(f"Run request: {RUN_REQUEST_ID}")
print(f"ax CLI: {shutil.which('ax') or 'not on PATH'}")
print(f"CLI credentials file: {'present' if CLI_CREDENTIALS_PATH.is_file() else 'missing'}")
print(
    "Query surface: "
    + (
        f"parquet {AX_PARQUET_DIR}"
        if AX_PARQUET_DIR
        else "ax experiment query CLI"
        if shutil.which("ax")
        else f"{AX_API_BASE}/api/v1/experiments/{EXPERIMENT_ID}/query"
    )
)

## Per-run metrics

Official AX `measurements` for this run request. `tool_calls_total` is the step
proxy. A run **passes** only when both `uri-reported` and
`connection-verified-in-agent` succeed.

In [ ]:
per_run_sql = f"""
SELECT
    run_id,
    agent,
    model,
    prompt_id,
    repeat_idx,
    status,
    exit_reason,
    test_pass_count,
    test_fail_count,
    agent_time_ms / 1000.0 AS runtime_s,
    tool_calls_total,
    tool_calls_failed,
    input_tokens,
    output_tokens,
    cost_usd_micros / 1000000.0 AS cost_usd
FROM measurements
WHERE run_request_id = '{RUN_REQUEST_ID}'
ORDER BY agent, prompt_id, repeat_idx
"""

runs = _with_database_column(ax_query(per_run_sql))
numeric_columns = [
    "repeat_idx",
    "test_pass_count",
    "test_fail_count",
    "runtime_s",
    "tool_calls_total",
    "tool_calls_failed",
    "input_tokens",
    "output_tokens",
    "cost_usd",
]
runs[numeric_columns] = runs[numeric_columns].apply(pd.to_numeric)
runs["both_tests_passed"] = runs["status"].eq("pass")
runs["database"] = pd.Categorical(runs["database"], categories=DATABASE_ORDER, ordered=True)
runs["agent"] = pd.Categorical(runs["agent"], categories=AGENT_ORDER, ordered=True)

print(f"Finished measurement rows: {len(runs)} (expected 18 if none starved)")
runs.head()

## Official test outcomes

Boolean pass/fail per test. Failure diagnostics are truncated platform messages,
not transcript text.

In [ ]:
tests_sql = f"""
SELECT
    agent,
    prompt_id,
    repeat_idx,
    test_name,
    exit_code = 0 AS passed,
    exit_code
FROM test_output
WHERE run_request_id = '{RUN_REQUEST_ID}'
ORDER BY agent, prompt_id, repeat_idx, test_name
"""

tests = _with_database_column(ax_query(tests_sql))
tests["repeat_idx"] = pd.to_numeric(tests["repeat_idx"])
tests["exit_code"] = pd.to_numeric(tests["exit_code"])
tests["passed"] = tests["passed"].astype(bool)
tests["database"] = pd.Categorical(tests["database"], categories=DATABASE_ORDER, ordered=True)
tests["agent"] = pd.Categorical(tests["agent"], categories=AGENT_ORDER, ordered=True)

test_summary = (
    tests.groupby(["agent", "database", "test_name"], observed=True)["passed"]
    .agg(passes="sum", n="count", rate="mean")
    .reset_index()
)
test_summary["rate_pct"] = test_summary["rate"] * 100
test_summary

In [ ]:
failures_sql = f"""
SELECT
    agent,
    prompt_id,
    repeat_idx,
    test_name,
    exit_code,
    stderr_tail
FROM test_output
WHERE run_request_id = '{RUN_REQUEST_ID}'
  AND exit_code != 0
ORDER BY agent, prompt_id, repeat_idx, test_name
"""

failures = _with_database_column(ax_query(failures_sql))
if failures.empty:
    print("No test failures on finished runs.")
else:
    display(failures)

## Summary by agent and database

In [ ]:
summary = (
    runs.groupby(["agent", "database"], observed=True)
    .agg(
        runs=("run_id", "size"),
        both_tests_passed=("both_tests_passed", "sum"),
        pass_rate=("both_tests_passed", "mean"),
        runtime_mean_s=("runtime_s", "mean"),
        tool_calls_mean=("tool_calls_total", "mean"),
        cost_total_usd=("cost_usd", "sum"),
        cost_mean_usd=("cost_usd", "mean"),
    )
    .reset_index()
)
summary["pass_rate_pct"] = summary["pass_rate"] * 100
summary["agent_label"] = summary["agent"].map(DISPLAY_NAMES)
summary["database_label"] = summary["database"].map(DISPLAY_NAMES)

totals = pd.Series(
    {
        "finished_runs": len(runs),
        "run_passes": int(runs["both_tests_passed"].sum()),
        "tests_passed": int(runs["test_pass_count"].sum()),
        "tests_failed": int(runs["test_fail_count"].sum()),
        "total_cost_usd": float(runs["cost_usd"].sum()),
        "mean_runtime_s": float(runs["runtime_s"].mean()),
        "mean_tool_calls": float(runs["tool_calls_total"].mean()),
    }
)

display(totals.to_frame("value").round(3))
summary[
    [
        "agent_label",
        "database_label",
        "runs",
        "both_tests_passed",
        "pass_rate_pct",
        "runtime_mean_s",
        "tool_calls_mean",
        "cost_mean_usd",
        "cost_total_usd",
    ]
].round(3)

## Visualize pass rate, runtime, cost, and steps

n=2 per cell, so treat rates as directional. The scatter keeps every finished run
visible instead of hiding variance behind averages.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle(
    "Database provisioning — docker-host smoke (3 agents × 3 DBs × n=2)",
    fontsize=15,
    fontweight="bold",
)

x = np.arange(len(DATABASE_ORDER))
width = 0.24


def grouped_bars(ax, column, title, ylabel, percent=False):
    for index, agent in enumerate(AGENT_ORDER):
        subset = summary[summary["agent"] == agent].set_index("database").reindex(DATABASE_ORDER)
        ax.bar(
            x + (index - 1) * width,
            subset[column].fillna(0),
            width=width,
            label=DISPLAY_NAMES[agent],
            color=COLORS[agent],
        )
    ax.set_title(title)
    ax.set_xlabel("Database")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x, [DISPLAY_NAMES[name] for name in DATABASE_ORDER])
    if percent:
        ax.set_ylim(0, 105)
    ax.legend(frameon=False, fontsize=8)


grouped_bars(axes[0, 0], "pass_rate_pct", "Both-tests pass rate", "Runs passing both tests (%)", percent=True)
grouped_bars(axes[0, 1], "runtime_mean_s", "Mean agent runtime", "Seconds")
grouped_bars(axes[1, 0], "cost_mean_usd", "Mean model cost", "USD per run")

for database in DATABASE_ORDER:
    subset = runs[runs["database"] == database]
    axes[1, 1].scatter(
        subset["tool_calls_total"],
        subset["runtime_s"],
        label=DISPLAY_NAMES[database],
        color=COLORS[database],
        s=70,
        alpha=0.85,
    )
axes[1, 1].set_title("Individual-run runtime versus tool calls")
axes[1, 1].set_xlabel("Tool calls (count)")
axes[1, 1].set_ylabel("Agent runtime (seconds)")
axes[1, 1].legend(title="Database", frameon=False)

plt.show()

## Docker guard and host-network recovery

Boolean flags only: whether the fail-fast Docker wrapper fired, and whether the
run later used `--network host`. No command text is returned.

In [ ]:
guard_sql = f"""
SELECT
    agent,
    prompt_id,
    countDistinct(run_id) AS runs,
    countDistinctIf(
        run_id,
        positionCaseInsensitive(payload, 'Docker bridge networking is unavailable in this sandbox.') > 0
    ) AS guard_triggered,
    countDistinctIf(
        run_id,
        positionCaseInsensitive(payload, '--network host') > 0
    ) AS used_host_network,
    countDistinctIf(
        run_id,
        positionCaseInsensitive(payload, 'docker run') > 0
    ) AS docker_run_attempted
FROM events
WHERE run_request_id = '{RUN_REQUEST_ID}'
  AND kind = 'tool_call'
GROUP BY agent, prompt_id
ORDER BY agent, prompt_id
"""

guard = _with_database_column(ax_query(guard_sql))
for column in ["runs", "guard_triggered", "used_host_network", "docker_run_attempted"]:
    guard[column] = pd.to_numeric(guard[column])
guard["database"] = pd.Categorical(guard["database"], categories=DATABASE_ORDER, ordered=True)
guard["agent"] = pd.Categorical(guard["agent"], categories=AGENT_ORDER, ordered=True)
guard

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
funnel = pd.Series(
    {
        "Finished runs": int(guard["runs"].sum()),
        "Attempted docker run": int(guard["docker_run_attempted"].sum()),
        "Guard triggered": int(guard["guard_triggered"].sum()),
        "Used --network host": int(guard["used_host_network"].sum()),
    }
)
bars = ax.barh(
    funnel.index,
    funnel.values,
    color=["#336791", "#8C8C8C", "#D97706", "#00A35C"],
)
ax.invert_yaxis()
ax.set_title("Docker networking path in the guarded smoke")
ax.set_xlabel("Runs (count)")
ax.set_xlim(0, max(18, funnel.max() + 1))
ax.bar_label(bars, padding=4)
plt.show()
funnel

## Interpretation and limitations

- Official AX scores are usable on this request: `ax-run-query` ran, and the
  lifecycle merge plus libSQL URI/operation support recovered the previous Claude
  and Codex false negatives.
- Cursor/SQLite repeat 1 starved before any agent events (15 minutes of no
  sandbox activity). It is a harness failure, not a SQLite provisioning failure,
  and it is absent from `measurements`.
- Remaining finished-run failures were: one readiness-only PostgreSQL check
  (`pg_isready` without SQL) and one `uri-reported` miss on a Claude PostgreSQL
  run whose final message did contain a URI. The latter is a transcript-surface
  issue (`stdout`/`transcript` vs final `message`), not a missing handoff.
- n=2 cannot rank databases. Efficiency comparisons should stay conditional on
  finished runs and should not treat starved harness rows as task failures.
- This still is not an independent exact-URI probe in the agent's final namespace.